In [1]:
# 기본
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
import gc
import glob
from collections import defaultdict
from statsmodels.stats.outliers_influence import variance_inflation_factor
import os

In [2]:
#컬럼별 상관계수 정렬하기
# 결과 저장용 DataFrame
result_df = pd.DataFrame()

# 201807 ~ 201812 반복
for month in range(7, 13):
    month_str = f"{month:02d}"
    file_path = f"corr_2018{month_str}.csv"

    # 파일 불러오기
    df = pd.read_csv(file_path, index_col=0)

    # segment 컬럼 추출 (보통 첫 번째 컬럼 이름이 'Segment'거나 비슷함)
    segment_corr = df['Segment']

    # 절댓값 기준 정렬 (NaN은 마지막에 유지)
    sorted_idx = segment_corr.abs().sort_values(ascending=False, na_position='last').index
    sorted_corr = segment_corr.loc[sorted_idx]

    # 월별 컬럼명과 상관계수로 저장
    temp_df = pd.DataFrame({
        f'{month_str}_컬럼': sorted_corr.index,
        f'{month_str}_상관계수': sorted_corr.values
    })

    # 결과 누적
    result_df = pd.concat([result_df, temp_df], axis=1)

# 결과 저장
result_df.to_csv("segment_corr_summary_with_nan.csv", index=False)
result_df


,07_컬럼,07_상관계수,08_컬럼,08_상관계수,09_컬럼,09_상관계수,10_컬럼,10_상관계수,11_컬럼,11_상관계수,12_컬럼,12_상관계수
0,Segment,1.000000,Segment,1.000000,Segment,1.000000,Segment,1.000000,Segment,1.000000,Segment,1.000000
1,정상청구원금_B5M,-0.690051,정상청구원금_B5M,-0.681754,정상청구원금_B5M,-0.669923,정상청구원금_B5M,-0.661894,정상청구원금_B5M,-0.649944,정상청구원금_B0M,-0.643568
2,정상청구원금_B2M,-0.636989,정상청구원금_B0M,-0.629619,정상청구원금_B0M,-0.638640,정상청구원금_B0M,-0.642101,정상청구원금_B0M,-0.642558,정상청구원금_B2M,-0.642101
3,정상청구원금_B0M,-0.613871,정상청구원금_B2M,-0.618813,정상청구원금_B2M,-0.613871,정상청구원금_B2M,-0.629619,정상청구원금_B2M,-0.638640,정상청구원금_B5M,-0.613871
4,정상입금원금_B5M,-0.571276,정상입금원금_B5M,-0.569619,이용금액_오프라인_B0M,-0.568058,이용금액_오프라인_B0M,-0.575217,이용금액_오프라인_R3M,-0.583078,이용금액_오프라인_R3M,-0.588755
...,...,...,...,...,...,...,...,...,...,...,...,...
204,이용금액_당사기타_B0M,NaN,이용건수_당사페이_B0M,NaN,신청건수_ATM_CL_B0,NaN,신청건수_ATM_CL_B0,NaN,승인거절건수_BL_B0M,NaN,신청건수_ATM_CL_B0,NaN
205,이용건수_당사페이_B0M,NaN,이용건수_당사기타_B0M,NaN,승인거절건수_입력오류_B0M,NaN,승인거절건수_입력오류_B0M,NaN,승인거절건수_입력오류_B0M,NaN,승인거절건수_입력오류_B0M,NaN
206,이용건수_당사기타_B0M,NaN,신청건수_ATM_CL_B0,NaN,승인거절건수_기타_B0M,NaN,승인거절건수_기타_B0M,NaN,승인거절건수_기타_B0M,NaN,승인거절건수_기타_B0M,NaN
207,승인거절건수_입력오류_B0M,NaN,승인거절건수_입력오류_B0M,NaN,승인거절건수_입력오류_R3M,NaN,승인거절건수_입력오류_R3M,NaN,승인거절건수_입력오류_R3M,NaN,승인거절건수_입력오류_R3M,NaN


In [10]:
# 병합할 데이터 리스트
all_data = []

# 201807 ~ 201812 반복
for month in range(7, 13):
    ym = f"2018{month:02d}"
    train_path = f"train/3.승인매출정보/{ym}_train_승인매출정보.parquet"

    try:
        # parquet에서 필요한 컬럼만 불러오기
        train_df = pd.read_parquet(train_path, columns=column_list)

        # object 또는 category 컬럼 인코딩하여 원래 컬럼 덮어쓰기
        obj_cols = train_df.select_dtypes(include=["object", "category"]).columns
        for col in obj_cols:
            train_df[col], _ = pd.factorize(train_df[col])

        # year_month 컬럼은 만들지 않음

        # 리스트에 추가
        all_data.append(train_df)

    except Exception as e:
        print(f"{ym} 불러오기 실패: {e}")
        continue

# 병합
all_df = pd.concat(all_data, axis=0, ignore_index=True)

In [11]:
print(all_df.shape)
all_df.info()

(2400000, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400000 entries, 0 to 2399999
Data columns (total 18 columns):
 #   Column         Dtype
---  ------         -----
 0   정상청구원금_B5M     int64
 1   정상청구원금_B2M     int64
 2   정상청구원금_B0M     int64
 3   정상입금원금_B5M     int64
 4   정상입금원금_B2M     int64
 5   정상입금원금_B0M     int64
 6   이용금액대          int64
 7   이용금액_온라인_R3M   int64
 8   이용금액_온라인_B0M   int64
 9   이용금액_오프라인_R6M  int64
 10  이용금액_오프라인_R3M  int64
 11  이용금액_오프라인_B0M  int64
 12  이용건수_오프라인_R6M  int64
 13  이용건수_오프라인_R3M  int64
 14  이용건수_오프라인_B0M  int64
 15  연체입금원금_B5M     int64
 16  연체입금원금_B2M     int64
 17  연체입금원금_B0M     int64
dtypes: int64(18)
memory usage: 329.6 MB


In [12]:
# 1. 수치형 컬럼만 추출 (이미 위 코드에서 이 단계까지 완료됨)
numeric_df = all_df.select_dtypes(include=["number"])

# 2. 결측치 제거 (VIF 계산 시 필수)
numeric_df = numeric_df.dropna()

# 3. VIF 계산 함수
def calculate_vif(df):
    vif_data = pd.DataFrame()
    vif_data["변수명"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif_data.sort_values(by="VIF", ascending=False)

# 4. VIF 계산 실행
vif_result = calculate_vif(numeric_df)

# 결과 확인
print(vif_result)

              변수명         VIF
2      정상청구원금_B0M  164.702724
1      정상청구원금_B2M  161.229502
0      정상청구원금_B5M  108.149116
5      정상입금원금_B0M   95.962869
4      정상입금원금_B2M   93.070641
3      정상입금원금_B5M   66.427571
13  이용건수_오프라인_R3M   50.929661
14  이용건수_오프라인_B0M   31.063116
12  이용건수_오프라인_R6M   30.297529
16     연체입금원금_B2M   22.349817
17     연체입금원금_B0M   21.204149
10  이용금액_오프라인_R3M   18.601336
15     연체입금원금_B5M   16.315530
11  이용금액_오프라인_B0M   14.622463
9   이용금액_오프라인_R6M   13.695711
8    이용금액_온라인_B0M    9.423475
7    이용금액_온라인_R3M    8.201633
6           이용금액대    1.290698


In [13]:
def reduce_vif(df, threshold=10.0, verbose=True):
    df = df.copy()
    variables = df.columns.tolist()
    
    while True:
        vif = [variance_inflation_factor(df[variables].values, i) for i in range(len(variables))]
        max_vif = max(vif)
        if max_vif > threshold:
            max_index = vif.index(max_vif)
            removed_feature = variables[max_index]
            if verbose:
                print(f"제거: {removed_feature} (VIF={max_vif:.2f})")
            variables.pop(max_index)
        else:
            break

    return df[variables]

In [14]:
# 수치형 데이터에서 VIF 기준 변수 제거
cleaned_df = reduce_vif(numeric_df, threshold=10)

# 최종 변수 목록 확인
print("최종 남은 변수들:")
print(cleaned_df.columns.tolist())

# 최종 VIF 확인
def calculate_vif(df):
    vif_data = pd.DataFrame()
    vif_data["변수명"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif_data.sort_values(by="VIF", ascending=False)

final_vif_result = calculate_vif(cleaned_df)
print(final_vif_result)


제거: 정상청구원금_B0M (VIF=164.70)
제거: 정상청구원금_B5M (VIF=92.85)
제거: 이용건수_오프라인_R3M (VIF=50.90)
제거: 정상청구원금_B2M (VIF=42.51)
제거: 이용건수_오프라인_B0M (VIF=20.16)
제거: 이용금액_오프라인_R3M (VIF=15.69)
제거: 정상입금원금_B0M (VIF=13.71)
최종 남은 변수들:
['정상입금원금_B5M', '정상입금원금_B2M', '이용금액대', '이용금액_온라인_R3M', '이용금액_온라인_B0M', '이용금액_오프라인_R6M', '이용금액_오프라인_B0M', '이용건수_오프라인_R6M', '연체입금원금_B5M', '연체입금원금_B2M', '연체입금원금_B0M']
              변수명       VIF
4    이용금액_온라인_B0M  9.235714
6   이용금액_오프라인_B0M  8.876002
1      정상입금원금_B2M  8.352138
3    이용금액_온라인_R3M  8.175647
5   이용금액_오프라인_R6M  7.798354
0      정상입금원금_B5M  6.754180
9      연체입금원금_B2M  5.458014
7   이용건수_오프라인_R6M  4.711417
10     연체입금원금_B0M  4.371559
8      연체입금원금_B5M  3.910221
2           이용금액대  1.217069


In [15]:
import pandas as pd
import os

# 추출할 컬럼 리스트
target_cols = [
    '이용개월수_온라인_R6M',
    '연속유실적개월수_기본_24M_카드',
    '이용금액대',
    '할부금액_무이자_3M_R12M'
]

# 공통 컬럼
base_cols = ['기준년월', 'ID']
final_cols = base_cols + target_cols

# 병합용 리스트
train_list = []
test_list = []

# 201807 ~ 201812 반복
for month in range(7, 13):
    ym = f"2018{month:02d}"

    try:
        # train
        train_path = f"train/3.승인매출정보/{ym}_train_승인매출정보.parquet"
        train_df = pd.read_parquet(train_path)
        train_filtered = train_df.loc[:, train_df.columns.intersection(final_cols)]
        train_list.append(train_filtered)

        # test
        test_path = f"test/3.승인매출정보/{ym}_test_승인매출정보.parquet"
        test_df = pd.read_parquet(test_path)
        test_filtered = test_df.loc[:, test_df.columns.intersection(final_cols)]
        test_list.append(test_filtered)

    except Exception as e:
        print(f"{ym} 처리 중 오류 발생: {e}")

# 병합
full_train = pd.concat(train_list, axis=0, ignore_index=True)
full_test = pd.concat(test_list, axis=0, ignore_index=True)

# 디렉토리 생성
os.makedirs("csv_output", exist_ok=True)

# 저장
full_train.to_csv("csv_output/3.승인매출_targetcols_train.csv", index=False)
full_test.to_csv("csv_output/3.승인매출_targetcols_test.csv", index=False)

print("✅ 선택된 컬럼 기준으로 train/test CSV 저장 완료!")

✅ 선택된 컬럼 기준으로 train/test CSV 저장 완료!


In [16]:
full_train.shape

(2400000, 6)